In [23]:
# ================== INICIALIZACIÓN COMPLETA DEL ENTORNO ==================
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import networkx as nx
import time
import requests

# Funciones de optimización TSP
def obtener_matriz_distancias_osrm(puntos):
    """
    Obtiene la matriz de distancias usando el servicio 'table' de OSRM para mayor eficiencia.
    Realiza una única llamada a la API en lugar de N*N llamadas.
    """
    n = len(puntos)
    if n <= 1:
        return [[0]] * n
    
    # Formato de coordenadas para el servicio 'table': {lon},{lat};{lon},{lat};...
    coords_str = ';'.join([f"{lon},{lat}" for lon, lat in puntos])
    url = f"http://router.project-osrm.org/table/v1/driving/{coords_str}?annotations=distance"
    
    try:
        response = requests.get(url, timeout=30) # Añadido timeout
        response.raise_for_status()
        data = response.json()
        
        if 'distances' in data:
            # OSRM devuelve distancias en metros, las convertimos a km
            distancias_km = [[dist / 1000 for dist in row] for row in data['distances']]
            return distancias_km
        else:
            return [[float('inf')]*n for _ in range(n)]
    except requests.exceptions.RequestException as e:
        print(f"Error al contactar OSRM: {e}")
        return [[float('inf')]*n for _ in range(n)]

def resolver_tsp(matriz):
    n = len(matriz)
    G = nx.Graph()
    for i in range(n):
        for j in range(n):
            if i != j:
                G.add_edge(i, j, weight=matriz[i][j])
    ciclo = nx.approximation.traveling_salesman_problem(G, weight='weight', cycle=True)
    distancia_total = sum(matriz[ciclo[i]][ciclo[i+1]] for i in range(len(ciclo)-1))
    return ciclo, distancia_total

# Carga de datos y widgets
ARCHIVO_DATOS = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Clientes_Ruta.xlsx'
df = pd.read_excel(ARCHIVO_DATOS, sheet_name=None)
df_clientes = df[list(df.keys())[0]]
df_cdr = df[list(df.keys())[1]]

# Mostrar cantidad de clientes por ruta
rutas_disponibles = sorted(df_clientes['Ruta'].dropna().unique())
clientes_por_ruta = {ruta: df_clientes[df_clientes['Ruta'] == ruta].shape[0] for ruta in rutas_disponibles}
rutas_labels = [f"{ruta} ({clientes_por_ruta[ruta]} clientes)" for ruta in rutas_disponibles]

# Widget de selección de rutas
rutas_widget = widgets.SelectMultiple(options=list(zip(rutas_labels, rutas_disponibles)), value=tuple(rutas_disponibles), description='Rutas a optimizar:', style={'description_width': 'initial'}, layout=widgets.Layout(width='60%'))

# Widget de selección de METs con casillas
mets_disponibles = sorted(df_cdr['CodMET'].dropna().unique())
met_checkboxes = [widgets.Checkbox(value=False, description=str(met), indent=False) for met in mets_disponibles]
met_box = widgets.VBox([widgets.Label('Selecciona uno o varios METs:')] + met_checkboxes)

# Widget para definir cantidad de clientes a procesar
clientes_a_procesar = widgets.IntText(value=10, description='Cantidad de clientes a procesar:', style={'description_width': 'initial'}, layout=widgets.Layout(width='30%'))

select_all_btn = widgets.Button(description='Seleccionar todo', button_style='info')
deselect_all_btn = widgets.Button(description='Deseleccionar todo', button_style='warning')
param_button = widgets.Button(description='Aplicar selección', button_style='success')
output_param = widgets.Output()

def seleccionar_todo(b):
    rutas_widget.value = tuple(rutas_disponibles)

def deseleccionar_todo(b):
    rutas_widget.value = tuple()

def aplicar_parametros(b):
    with output_param:
        clear_output()
        mets_seleccionados = [met.description for met in met_checkboxes if met.value]
        print(f'MET(s) seleccionados: {mets_seleccionados}')
        rutas_seleccionadas = list(rutas_widget.value)
        print(f'Rutas a optimizar: {rutas_seleccionadas}')
        print(f'Cantidad de clientes a procesar: {clientes_a_procesar.value}')

select_all_btn.on_click(seleccionar_todo)
deselect_all_btn.on_click(deseleccionar_todo)
param_button.on_click(aplicar_parametros)

# Mostrar el display interactivo
display(widgets.VBox([met_box, rutas_widget, clientes_a_procesar, widgets.HBox([select_all_btn, deselect_all_btn]), param_button, output_param]))
# Los parámetros seleccionados estarán en met_checkboxes, rutas_widget.value y clientes_a_procesar.value para usarse en la optimización.

In [24]:
# ================== MAPA Y EXCEL ÚNICO CON CAPAS POR MET Y RUTA ==================
import openpyxl
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter
import os
from folium.features import CustomIcon, DivIcon
from datetime import datetime
import folium
import random

output_dir = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Outputs'
os.makedirs(output_dir, exist_ok=True)
icon_met_path = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\95_24.png'
mets_seleccionados = [met.description for met in met_checkboxes if met.value]
rutas_seleccionadas = list(rutas_widget.value)
num_clientes = clientes_a_procesar.value
fecha_hora = datetime.now().strftime('%Y%m%d_%H%M%S')

# Inicializar mapa centrado en el primer MET
if mets_seleccionados:
    met_row = df_cdr[df_cdr['CodMET'] == mets_seleccionados[0]].iloc[0]
    mapa = folium.Map(location=[met_row['U_latitud'], met_row['U_longitud']], zoom_start=10, tiles='OpenStreetMap')
else:
    mapa = folium.Map(location=[20.5, -100.8], zoom_start=7, tiles='OpenStreetMap')

colores = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen', 'gray', 'black', 'lightgray']

export_rows = []
resumen_rutas = []
total_clientes = 0
distancia_total = 0
max_dist_general = 0
max_from_general = ''
max_to_general = ''

# Crear FeatureGroups por MET y por ruta
featuregroups_met = {}
featuregroups_ruta = {}
# Guardar combinaciones MET-Ruta para filtrado dinámico
combinaciones_met_ruta = set()
# En la generación de IDs para MET y ruta, asegúrate de normalizar correctamente los nombres para que coincidan en el JS
def normalizar_id(valor):
    return str(valor).replace(' ', '').replace(':', '').replace('.', '').replace('á','a').replace('é','e').replace('í','i').replace('ó','o').replace('ú','u').replace('ñ','n')
for idx_met, met in enumerate(mets_seleccionados):
    met_row = df_cdr[df_cdr['CodMET'] == met].iloc[0]
    met_id = normalizar_id(met)
    fg_met = folium.FeatureGroup(name=f"MET {met}", show=True if idx_met==0 else False)
    featuregroups_met[met] = fg_met
    for idx_ruta, ruta in enumerate(rutas_seleccionadas):
        ruta_id = normalizar_id(ruta)
        clientes_ruta = df_clientes[df_clientes['Ruta'] == ruta].head(num_clientes)
        if clientes_ruta.empty:
            continue
        combinaciones_met_ruta.add((met_id, ruta_id))
        puntos = [(met_row['U_longitud'], met_row['U_latitud'])] + [(row['U_longitud'], row['U_latitud']) for _, row in clientes_ruta.iterrows()]
        codigos = [met] + list(clientes_ruta['CodCli'])
        matriz = obtener_matriz_distancias_osrm(puntos)
        ciclo, distancia_total_ruta = resolver_tsp(matriz)
        ciclo_filtrado = []
        vistos = set()
        for idx, val in enumerate(ciclo):
            if idx == 0 or idx == len(ciclo)-1 or val not in vistos:
                ciclo_filtrado.append(val)
                vistos.add(val)
        if ciclo_filtrado[-1] != 0:
            ciclo_filtrado.append(0)
        recorrido_codigos = [codigos[i] for i in ciclo_filtrado]

        distancias = []
        max_dist = 0
        max_from = ''
        max_to = ''
        for i in range(len(ciclo_filtrado)-1):
            d = matriz[ciclo_filtrado[i]][ciclo_filtrado[i+1]]
            distancias.append(d)
            if d > max_dist:
                max_dist = d
                max_from = codigos[ciclo_filtrado[i]]
                max_to = codigos[ciclo_filtrado[i+1]]

        # Resumen por ruta
        total_clientes_ruta = len(clientes_ruta)
        distancia_promedio_ruta = distancia_total_ruta / total_clientes_ruta if total_clientes_ruta > 0 else 0
        resumen_rutas.append({
            'MET': met,
            'Ruta': ruta,
            'Clientes': total_clientes_ruta,
            'Distancia_total_km': distancia_total_ruta,
            'Distancia_promedio_cliente_km': distancia_promedio_ruta,
            'Distancia_maxima_km': max_dist,
            'De': max_from,
            'A': max_to
        })
        total_clientes += total_clientes_ruta
        distancia_total += distancia_total_ruta
        if max_dist > max_dist_general:
            max_dist_general = max_dist
            max_from_general = max_from
            max_to_general = max_to

        # Exportar filas para Excel
        for idx, sec in enumerate(ciclo_filtrado):
            distancia_val = distancias[idx] if idx < len(distancias) else ''
            if sec == 0:
                export_rows.append({
                    'MET': met,
                    'Ruta': ruta,
                    'Secuencia': idx+1,
                    'Tipo': 'MET',
                    'Codigo': met,
                    'Longitud': met_row['U_longitud'],
                    'Latitud': met_row['U_latitud'],
                    'Nombre': met_row.get('Nombre', ''),
                    'Distancia_al_siguiente_km': distancia_val,
                    'Recorrido': ' -> '.join([str(x) for x in recorrido_codigos]),
                    'Distancia_total_km': distancia_total_ruta
                })
            else:
                cli_row = clientes_ruta.iloc[sec-1]
                export_rows.append({
                    'MET': met,
                    'Ruta': ruta,
                    'Secuencia': idx+1,
                    'Tipo': 'Cliente',
                    'Codigo': cli_row['CodCli'],
                    'Longitud': cli_row['U_longitud'],
                    'Latitud': cli_row['U_latitud'],
                    'Nombre': cli_row.get('Nombre', ''),
                    'Distancia_al_siguiente_km': distancia_val,
                    'Recorrido': ' -> '.join([str(x) for x in recorrido_codigos]),
                    'Distancia_total_km': distancia_total_ruta
                })

        # Crear FeatureGroup por ruta si no existe
        if ruta not in featuregroups_ruta:
            featuregroups_ruta[ruta] = folium.FeatureGroup(name=f"Ruta {ruta}", show=False)

        # Dibujar recorrido en el mapa en ambas capas
        ruta_coords = [puntos[i][::-1] for i in ciclo_filtrado]
        color_ruta = colores[idx_ruta % len(colores)]
        folium.PolyLine(ruta_coords, color=color_ruta, weight=5, opacity=0.8, tooltip=f"MET {met} - Ruta {ruta} | Distancia: {distancia_total_ruta:.2f} km",
            **{'data-met': met_id, 'data-ruta': ruta_id}
        ).add_to(featuregroups_met[met])
        folium.PolyLine(ruta_coords, color=color_ruta, weight=5, opacity=0.8, tooltip=f"MET {met} - Ruta {ruta} | Distancia: {distancia_total_ruta:.2f} km",
            **{'data-met': met_id, 'data-ruta': ruta_id}
        ).add_to(featuregroups_ruta[ruta])

        # Marcadores de clientes y secuencia en ambas capas
        secuencia_cliente = 1
        for idx, sec in enumerate(ciclo_filtrado[1:]):
            if sec == 0:
                continue
            cli_row = clientes_ruta.iloc[sec-1]
            codcli = cli_row['CodCli']
            nombre = cli_row.get('Nombre', '')
            ruta_cliente = cli_row.get('Ruta', '')
            idx_ciclo = idx + 1
            distancia_anterior = matriz[ciclo_filtrado[idx_ciclo-1]][ciclo_filtrado[idx_ciclo]] if idx_ciclo > 0 else 'N/A'
            distancia_siguiente = matriz[ciclo_filtrado[idx_ciclo]][ciclo_filtrado[idx_ciclo+1]] if idx_ciclo+1 < len(ciclo_filtrado) else 'N/A'

            popup_html = f'''
            <div style=\"background: #fff; border-radius: 16px; box-shadow: 0 2px 8px rgba(0,0,0,0.15); padding: 14px; border: 2px solid #8BC34A; min-width: 260px; max-width: 340px;\">
                <div style=\"font-size:20px; color:{color_ruta}; font-weight:bold; margin-bottom:4px;\">
                    <span style=\"vertical-align:middle;\">👨‍💼</span> Cliente: <span style=\"color:{color_ruta};\">{codcli}</span>
                </div>
                <div style=\"font-size:16px; color:#333; margin-bottom:4px;\">
                    <span style=\"vertical-align:middle;\">📚</span> <b>Nombre:</b> {nombre}
                </div>
                <div style=\"font-size:15px; color:#2A81CB; margin-bottom:4px;\">
                    <span style=\"vertical-align:middle;\">🗺️</span> <b>Ruta:</b> {ruta_cliente}
                </div>
                <div style=\"font-size:15px; color:#CB2A2A; margin-bottom:4px;\">
                    <span style=\"vertical-align:middle;\">↩️</span> <b>Distancia anterior:</b> {distancia_anterior if isinstance(distancia_anterior, float) else 'N/A'} km
                </div>
                <div style=\"font-size:15px; color:#2A81CB; margin-bottom:4px;\">
                    <span style=\"vertical-align:middle;\">➡️</span> <b>Distancia siguiente:</b> {distancia_siguiente if isinstance(distancia_siguiente, float) else 'N/A'} km
                </div>
                <div style=\"font-size:15px; color:#FFC107; margin-bottom:4px;\">
                    <span style=\"vertical-align:middle;\">🔢</span> <b>Número de secuencia:</b> {secuencia_cliente}
                </div>
                <div style=\"font-size:14px; color:#333; margin-bottom:2px;\">MET: <b>{met}</b></div>
            </div>
            '''
            folium.Marker(
                location=[cli_row['U_latitud'], cli_row['U_longitud']],
                popup=folium.Popup(popup_html, max_width=340),
                icon=folium.Icon(color=color_ruta, icon='info-sign')
            ).add_to(featuregroups_met[met])
            folium.Marker(
                location=[cli_row['U_latitud'], cli_row['U_longitud']],
                popup=folium.Popup(popup_html, max_width=340),
                icon=folium.Icon(color=color_ruta, icon='info-sign')
            ).add_to(featuregroups_ruta[ruta])
            folium.Marker(
                location=[cli_row['U_latitud'], cli_row['U_longitud']],
                icon=DivIcon(
                    icon_size=(24,24),
                    icon_anchor=(12,12),
                    html=f'<div style="font-size:16px; color:white; background:{color_ruta}; border-radius:50%; width:24px; height:24px; text-align:center; line-height:24px; border:2px solid #fff;">{secuencia_cliente}</div>'
                ),
            ).add_to(featuregroups_met[met])
            secuencia_cliente += 1

    # MET marker en su capa
    folium.Marker(location=[met_row['U_latitud'], met_row['U_longitud']],
                  popup=folium.Popup(f'''<div style="background:#2A81CB; color:#fff; border-radius:14px; box-shadow:0 2px 8px rgba(0,0,0,0.15); padding:14px; border:2px solid #fff; min-width:220px; text-align:center;">
                    <div style="font-size:20px; font-weight:bold; margin-bottom:6px;">🏠 MET: <span style="color:#FFD600;">{met}</span></div>
                    <div style="font-size:15px; color:#fff; margin-bottom:2px;">Coordenadas:<br><span style="font-size:14px; color:#FFD600;">{met_row['U_latitud']}, {met_row['U_longitud']}</span></div>
                  </div>''', max_width=260),
                  icon=CustomIcon(icon_met_path, icon_size=(32,32))).add_to(featuregroups_met[met])

# Agregar todas las capas al mapa
for fg in featuregroups_met.values():
    fg.add_to(mapa)
for fg in featuregroups_ruta.values():
    fg.add_to(mapa)
folium.LayerControl(collapsed=False, autoZIndex=True).add_to(mapa)

# Mover el control de capas debajo del control de zoom
move_layers_control_js = '''
<script>
document.addEventListener('DOMContentLoaded', function() {
    var zoomControl = document.querySelector('.leaflet-control-zoom');
    var layersControl = document.querySelector('.leaflet-control-layers');
    if (zoomControl && layersControl) {
        zoomControl.parentNode.insertBefore(layersControl, zoomControl.nextSibling);
    }
});
</script>
'''
mapa.get_root().html.add_child(folium.Element(move_layers_control_js))

# Título centrado arriba
titulo_html = f'''<div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 9998; background-color: white; padding: 7px 20px; border: 2px solid #333; border-radius: 12px; box-shadow: 0 2px 8px rgba(0,0,0,0.3);"><h2 style="margin: 0; color: #333; text-align: center; font-size:14px;"><span style='font-size:14px;'>🗺️ MAPA RECORRIDOS ÓPTIMOS METS-CLIENTE</span></h2><p style="margin: 4px 0 0 0; text-align: center; color: #666; font-size: 9px;">METS analizados: <b>{len(mets_seleccionados)}</b> | Rutas: <b>{len(rutas_seleccionadas)}</b></p></div>'''
mapa.get_root().html.add_child(folium.Element(titulo_html))

# Calcular resumen por MET
resumen_mets = {}
for met in mets_seleccionados:
    rutas_met = [r for r in resumen_rutas if r['MET'] == met]
    total_clientes_met = sum(r['Clientes'] for r in rutas_met)
    distancia_total_met = sum(r['Distancia_total_km'] for r in rutas_met)
    distancia_promedio_met = distancia_total_met / total_clientes_met if total_clientes_met > 0 else 0
    max_ruta = max(rutas_met, key=lambda x: x['Distancia_maxima_km'], default=None)
    max_dist_met = max_ruta['Distancia_maxima_km'] if max_ruta else 0
    max_from_met = max_ruta['De'] if max_ruta else ''
    max_to_met = max_ruta['A'] if max_ruta else ''
    resumen_mets[met] = {
        'total_clientes': total_clientes_met,
        'distancia_total': distancia_total_met,
        'distancia_promedio': distancia_promedio_met,
        'max_dist': max_dist_met,
        'max_from': max_from_met,
        'max_to': max_to_met,
        'rutas': rutas_met
    }
# Resumen general a la esquina superior derecha, ahora por MET
resumen_html = (
    f'<div id="resumen-general" style="position: fixed; top: 20px; right: 35px; width: 340px; background-color: white; border: 2px solid #333; z-index: 9997; font-size: 11px; padding: 14px; border-radius: 12px; box-shadow: 0 2px 8px rgba(0,0,0,0.3);">'
    )
for idx_met, (met, resumen_met) in enumerate(resumen_mets.items()):
    met_id = normalizar_id(met)
    color_met = colores[idx_met % len(colores)]
    resumen_html += (
        f'<div id="resumen-{met_id}" class="resumen-met" style="display:block; margin-bottom:10px;">'
        f'<h3 style="margin-top: 0; color: {color_met}; text-align: left; border-bottom: 2px solid #ddd; padding-bottom: 5px; font-size:14px;"><span style="font-size:14px;">🏠 MET {met}</span></h3>'
        f'<p style="margin: 5px 0; font-weight: bold; color: #333; font-size:12px;">🧮 Total clientes: {resumen_met["total_clientes"]}</p>'
        f'<p style="margin: 6px 0 4px 0; color: #444; font-size:11px;">🚗 <b>Distancia total:</b> {resumen_met["distancia_total"]:.2f} km</p>'
        f'<p style="margin: 6px 0 4px 0; color: #444; font-size:11px;">📏 <b>Distancia promedio por cliente:</b> {resumen_met["distancia_promedio"]:.2f} km</p>'
        f'<p style="margin: 6px 0 4px 0; color: #CB2A2A; font-size:11px;">🏆 <b>Distancia máxima:</b> {resumen_met["max_dist"]:.2f} km <b>De:</b> {resumen_met["max_from"]} <b>A:</b> {resumen_met["max_to"]}</p>'
        f'<hr style="margin:8px 0;"><span style="font-size:13px;">🔎 <b>Resumen por ruta:</b></span>'
    )
    for idx_ruta, resumen in enumerate(resumen_met['rutas']):
        ruta_id = normalizar_id(resumen['Ruta'])
        color_ruta = colores[idx_ruta % len(colores)]
        capa_id = f"resumen-{met_id}-{ruta_id}"
        resumen_html += (
            f'<div id="{capa_id}" class="resumen-capa" style="display:block;">'
            f'<p style="margin: 4px 0 0 0; color: {color_ruta}; font-size:11px;">MET {met} 🛣️ Ruta {resumen["Ruta"]}: {resumen["Clientes"]} clientes, Distancia: {resumen["Distancia_total_km"]:.2f} km, Promedio por cliente: {resumen["Distancia_promedio_cliente_km"]:.2f} km, <span style="color:#CB2A2A;">🏆 Máxima: {resumen["Distancia_maxima_km"]:.2f} km de {resumen["De"]} a {resumen["A"]}</span></p>'
            f'</div>'
        )
    resumen_html += "</div>"
resumen_html += "</div>"
mapa.get_root().html.add_child(folium.Element(resumen_html))
# JavaScript para mostrar/ocultar resumen por MET y por ruta según capas activas
update_resumen_js = '''
<script>
function normalizarId(valor) { // Corregido para coincidir con la lógica de Python y eliminar warnings
    if (typeof valor !== 'string') valor = String(valor);
    return valor.replace(/\s|:|\./g, '').replace(/[áÁ]/g,'a').replace(/[éÉ]/g,'e').replace(/[íÍ]/g,'i').replace(/[óÓ]/g,'o').replace(/[úÚ]/g,'u').replace(/[ñÑ]/g,'n');
}
document.addEventListener('DOMContentLoaded', function() {
    function updateResumen() {
        var metActivos = {};
        var rutaActivos = {};
        var overlays = document.querySelectorAll('.leaflet-control-layers-overlays label');
        overlays.forEach(function(label) {
            var input = label.querySelector('input');
            var span = label.querySelector('span:last-child');
            if (!span) return;
            var text = span.textContent.trim();
            if (text.startsWith('MET')) {
                var metId = normalizarId(text.replace(/MET\\s*/,''));
                metActivos[metId] = input.checked;
            } else if (text.startsWith('Ruta')) {
                var rutaId = normalizarId(text.replace(/Ruta\\s*/,''));
                rutaActivos[rutaId] = input.checked;
            }
        });
        var resumenMetDivs = document.querySelectorAll('.resumen-met');
        resumenMetDivs.forEach(function(div) {
            var metMatch = div.id.match(/resumen-(.+)/);
            if (metMatch) {
                var met = metMatch[1];
                var rutasDivs = div.querySelectorAll('.resumen-capa');
                var algunaRutaActiva = false;
                rutasDivs.forEach(function(rutaDiv) {
                    var idMatch = rutaDiv.id.match(/resumen-(.+)-(.+)/);
                    if (idMatch) {
                        var ruta = idMatch[2];
                        if (rutaActivos[ruta]) {
                            algunaRutaActiva = true;
                        }
                        rutaDiv.style.display = (metActivos[met] && rutaActivos[ruta]) ? 'block' : 'none';
                    }
                });
                div.style.display = metActivos[met] ? 'block' : 'none';
            }
        });
        var polylines = document.querySelectorAll('.leaflet-interactive');
        polylines.forEach(function(line) {
            var met = line.getAttribute('data-met');
            var ruta = line.getAttribute('data-ruta');
            if (met && ruta) {
                line.style.display = (metActivos[met] && rutaActivos[ruta]) ? 'block' : 'none';
            }
        });
    }
    var overlays = document.querySelectorAll('.leaflet-control-layers-overlays input');
    overlays.forEach(function(input) {
        input.addEventListener('change', updateResumen);
    });
    updateResumen();
});
</script>
'''
mapa.get_root().html.add_child(folium.Element(update_resumen_js))

nombre_mapa = os.path.join(output_dir, f'mapa_recorridos_optimos_todos_{fecha_hora}.html')
mapa.save(nombre_mapa)
print(f'Mapa grupal exportado a: {nombre_mapa}')

# Exportar Excel único con formato profesional
import pandas as pd
df_export = pd.DataFrame(export_rows)
df_resumen = pd.DataFrame(resumen_rutas)
resumen_general = pd.DataFrame([{
    'Total_clientes': total_clientes,
    'Distancia_total_km': distancia_total,
    'Distancia_promedio_cliente_km': distancia_total / total_clientes if total_clientes > 0 else 0,
    'Distancia_maxima_km': max_dist_general,
    'De': max_from_general,
    'A': max_to_general
}])
excel_path = os.path.join(output_dir, f'recorridos_optimos_todos_{fecha_hora}.xlsx')
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_export.to_excel(writer, sheet_name='Recorridos', index=False)
    df_resumen.to_excel(writer, sheet_name='Resumen por ruta', index=False)
    resumen_general.to_excel(writer, sheet_name='Resumen general', index=False)

# Formatear el Excel con openpyxl
wb = openpyxl.load_workbook(excel_path)
header_font = Font(bold=True, color='FFFFFF', size=12)
header_fill = PatternFill('solid', fgColor='4F81BD')
cell_fill = PatternFill('solid', fgColor='DDEBF7')
border = Border(left=Side(style='thin', color='4F81BD'), right=Side(style='thin', color='4F81BD'), top=Side(style='thin', color='4F81BD'), bottom=Side(style='thin', color='4F81BD'))
align = Alignment(horizontal='center', vertical='center', wrap_text=True)

for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]
    for col_idx, cell in enumerate(ws[1], 1):
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = align
        cell.border = border
        if 'Ruta' in cell.value: cell.value = '🛣️ ' + str(cell.value)
        if 'Cliente' in cell.value: cell.value = '👨‍💼 ' + str(cell.value)
        if 'Distancia' in cell.value: cell.value = '📏 ' + str(cell.value)
        if 'Secuencia' in cell.value: cell.value = '🔢 ' + str(cell.value)
        if 'MET' in cell.value: cell.value = '🏠 ' + str(cell.value)
        if 'Nombre' in cell.value: cell.value = '📚 ' + str(cell.value)
        if 'De' == cell.value: cell.value = '🔴 De'
        if 'A' == cell.value: cell.value = '🟢 A'

    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.fill = cell_fill
            cell.alignment = align
            cell.border = border

    for col in ws.columns:
        max_length = 0
        col_letter = get_column_letter(col[0].column)
        for cell in col:
            try:
                if cell.value:
                    max_length = max(max_length, len(str(cell.value)))
            except:
                pass
        ws.column_dimensions[col_letter].width = max(12, min(max_length + 2, 40))

    if sheet_name in ['Resumen general', 'Resumen por ruta']:
        for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
            for cell in row:
                cell.fill = PatternFill('solid', fgColor='FFD966')

wb.save(excel_path)
print(f'Excel grupal exportado a: {excel_path}')

<>:289: SyntaxWarning: invalid escape sequence '\.'
<>:289: SyntaxWarning: invalid escape sequence '\.'
C:\Users\igcamposg\AppData\Local\Temp\ipykernel_9828\1478009824.py:289: SyntaxWarning: invalid escape sequence '\.'
  return valor.replace(/\\s|:|\./g, '').replace(/[áÁ]/g,'a').replace(/[éÉ]/g,'e').replace(/[íÍ]/g,'i').replace(/[óÓ]/g,'o').replace(/[úÚ]/g,'u').replace(/[ñÑ]/g,'n');


<>:289: SyntaxWarning: invalid escape sequence '\.'
<>:289: SyntaxWarning: invalid escape sequence '\.'
C:\Users\igcamposg\AppData\Local\Temp\ipykernel_9828\1478009824.py:289: SyntaxWarning: invalid escape sequence '\.'
  return valor.replace(/\\s|:|\./g, '').replace(/[áÁ]/g,'a').replace(/[éÉ]/g,'e').replace(/[íÍ]/g,'i').replace(/[óÓ]/g,'o').replace(/[úÚ]/g,'u').replace(/[ñÑ]/g,'n');


Mapa grupal exportado a: C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Outputs\mapa_recorridos_optimos_todos_20250901_144159.html
Excel grupal exportado a: C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Outputs\recorridos_optimos_todos_20250901_144159.xlsx
